# Silver Layer Quality & Reconciliation Framework
**Objective:** To perform a "Source-to-Target" reconciliation and validate Business Rule enforcement in the Silver Layer.

### Audit Checklist:
1. **Row-Level Reconciliation**: $Bronze Total = Silver Valid + Silver Quarantine + Dropped Duplicates$.
2. **Type Casting Integrity**: Ensuring `price_rub` and `listing_id` are not NULL after `try_cast`.
3. **Business Rule Enforcement**: Verifying that `price_usd` is correctly calculated and `car_age_years` is logical.

#Setup & Environment Configuration

In [0]:
# Section 1: Setup & Target Discovery
from pyspark.sql.functions import col, count, sum, abs, upper

# 1. SETUP WIDGETS
dbutils.widgets.text("project_catalog", "vstone_catalog")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("bronze_schema", "bronze")

CAT_NAME = dbutils.widgets.get("project_catalog")
SILVER_PATH = f"{CAT_NAME}.{dbutils.widgets.get('silver_schema')}"
BRONZE_PATH = f"{CAT_NAME}.{dbutils.widgets.get('bronze_schema')}"

print(f" AUDITING SILVER LAYER: {SILVER_PATH}")
print("="*60)

# Source-to-Target Reconciliation Logic

In [0]:
# Section 2: Global Reconciliation Logic

# 1. Load ALL Bronze Tables to get the True Total Ingested
cnt_bronze_1 = spark.table(f"{BRONZE_PATH}.listings_csv_copyinto").count()
cnt_bronze_2 = spark.table(f"{BRONZE_PATH}.listings_csv_dlt").count()
cnt_bronze_3 = spark.table(f"{BRONZE_PATH}.listings_json_autoloader").count()
cnt_bronze_4 = spark.table(f"{BRONZE_PATH}.listings_xml_pyspark").count()

# Total incoming rows from all 4 chunks
cnt_bronze_total = cnt_bronze_1 + cnt_bronze_2 + cnt_bronze_3 + cnt_bronze_4

# 2. Load Fully Merged Silver & Quarantine Tables
cnt_silver = spark.table(f"{SILVER_PATH}.listings_silver_merged").count()
cnt_quarantine = spark.table(f"{SILVER_PATH}.listings_main_quarantine").count()

# 3. Duplicate Check
# Total Ingested (Bronze) - (Clean Data + Bad Data) = Duplicates dropped
deduped_diff = cnt_bronze_total - (cnt_silver + cnt_quarantine)

print(f" RECONCILIATION SUMMARY:")
print(f"   - Total Ingested (All Bronze) : {cnt_bronze_total:,}")
print(f"   - Valid Records (Silver)      : {cnt_silver:,}")
print(f"   - Rejected (Quarantine)       : {cnt_quarantine:,}")
print(f"   - Identified Duplicates       : {deduped_diff:,}")

# 4. Strict Validation
# Ensure we don't have negative duplicates and the equation perfectly balances
if deduped_diff >= 0 and (cnt_silver + cnt_quarantine + deduped_diff) == cnt_bronze_total:
    print("    [PASS] Strict Reconciliation: Math matches perfectly. Zero data loss!")
else:
    print("    [FAIL] Reconciliation: Data mismatch detected!")
    # Fail the pipeline immediately so bad data doesn't move forward
    raise ValueError(f"Reconciliation Failed! Bronze: {cnt_bronze_total}, Silver+Quarantine: {cnt_silver + cnt_quarantine}")

# Dynamic Schema & Null Integrity Testing

In [0]:
from pyspark.sql.functions import col, count

# 1. DEFINE VARIABLES (Dynamic Approach)
CATALOG_NAME = dbutils.widgets.get("project_catalog")
SCHEMA_NAME = dbutils.widgets.get("silver_schema")
FULL_PATH = f"{CATALOG_NAME}.{SCHEMA_NAME}"

print(f" STARTING DYNAMIC TESTING FOR: {FULL_PATH}")
print("="*60)

# 2. GET TABLE LIST DYNAMICALLY (No Hardcoding)
tables_list = [row['tableName'] for row in spark.sql(f"SHOW TABLES IN {FULL_PATH}").collect()]

# 3. LOOP THROUGH EACH TABLE FOR TESTING
for t_name in tables_list:
    full_table_path = f"{FULL_PATH}.{t_name}"
    
    try:
        # Table load karein
        df = spark.table(full_table_path)
        row_count = df.count()
        
        print(f"\n TABLE: {t_name.upper()}")
        print(f"   - Total Rows: {row_count:,}")

        # --- TEST 1: DATA AVAILABILITY ---
        if row_count > 0:
            print("    Status: Passed (Data exists)")
        else:
            print("    Status: Warning (Table is empty)")

        # --- TEST 2: NULL INTEGRITY (Dynamic Column Detection) ---
        check_col = "listing_id" if "listing_id" in df.columns else df.columns[0]
        nulls = df.filter(col(check_col).isNull()).count()
        
        if nulls == 0:
            print(f"    Integrity: 0 NULLs in '{check_col}'")
        else:
            print(f"    Integrity: Found {nulls} NULLs in '{check_col}'!")

        # --- TEST 3: QUARANTINE LOGIC CHECK ---
        if "quarantine" in t_name.lower():
            if "quarantine_reason" in df.columns:
                print("    Audit: 'quarantine_reason' column is present.")
            else:
                print("    Audit: Missing 'quarantine_reason' in Quarantine table!")

    except Exception as e:
        print(f"    ERROR: Could not process {t_name}: {str(e)}")

print("\n" + "="*60 + "\n ALL TABLES TESTED SUCCESSFULLY ")

#Business Logic Verification (Currency Math)

In [0]:
from pyspark.sql.functions import col, abs as spark_abs

print("\n STARTING TEST 3: CURRENCY NORMALIZATION ACCURACY")

# 1. Validation Logic using Spark
tolerance = 0.01
math_errors = df_silver.filter(col("price_rub") > 0) \
    .withColumn("diff", spark_abs(col("price_usd") - (col("price_rub") / 82.5))) \
    .filter(col("diff") > tolerance)

error_count = math_errors.count()

# 2. Results Output
if error_count == 0:
    print(f" PASSED: Currency Normalization is 100% accurate.")
    
    sample = df_silver.select("price_rub", "price_usd").filter(col("price_rub") > 0).first()
    
    rub_val = float(sample['price_rub'])
    usd_val = float(sample['price_usd'])
    manual_calc = rub_val / 82.5
    
    print(f"   Sample Verification Details:")
    print(f"   - Input RUB: {rub_val:,.2f}")
    print(f"   - Converted USD (in Table): {usd_val:,.2f}")
    print(f"   - Manual Formula Check: {rub_val:,.2f} / 82.5 = {manual_calc:,.2f}")
else:
    print(f" FAILED: Found {error_count} records with calculation mismatch!")
    math_errors.select("listing_id", "price_rub", "price_usd", "diff").show(5)

print("-" * 50)

# Ecosystem-Wide Master Audit

In [0]:
CATALOG_NAME = dbutils.widgets.get("project_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

print(f" DYNAMIC MASTER AUDIT: {CATALOG_NAME} ECOSYSTEM")
print("="*80)

def get_table_counts(schema_name, filter_keyword=None, exclude_keyword=None):
    tables_df = spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{schema_name}")
    table_list = [row['tableName'] for row in tables_df.collect()]
    total_count, scanned_tables = 0, []
    
    for t in table_list:
        if filter_keyword and filter_keyword not in t: continue
        if exclude_keyword and exclude_keyword in t: continue
            
        full_path = f"{CATALOG_NAME}.{schema_name}.{t}"
        c = spark.table(full_path).count()
        total_count += c
        scanned_tables.append(f"{t} ({c:,})")
    return total_count, scanned_tables

try:
    total_bronze, bronze_list = get_table_counts(BRONZE_SCHEMA)
    total_quarantine, quarantine_list = get_table_counts(SILVER_SCHEMA, filter_keyword="quarantine")
    total_silver, silver_list = get_table_counts(SILVER_SCHEMA, exclude_keyword="quarantine")
    duplicates = total_bronze - (total_silver + total_quarantine)

    print(f" TOTAL BRONZE (Scanned {len(bronze_list)} Tables): {total_bronze:,}")
    print(f" TOTAL QUARANTINE (Scanned {len(quarantine_list)} Tables): {total_quarantine:,}")
    print(f" TOTAL SILVER CLEAN (Scanned {len(silver_list)} Tables): {total_silver:,}")
    print(f" CALCULATED DUPLICATES: {duplicates:,}")
    
    if total_bronze == (total_silver + total_quarantine + duplicates):
        print(" DYNAMIC RECONCILIATION SUCCESS: 100% Data Accounted For!")
except Exception as e:
    print(f" DYNAMIC AUDIT ERROR: {str(e)}")

# ACID Compliance & Time Travel Audit

In [0]:
from pyspark.sql.functions import col

print(f"\n{'='*70}")
print(f" TEST 7: ADVANCED DELTA HISTORY & AUDIT TRAIL")
print(f"{'='*70}")

TABLE_PATH = f"{CAT_NAME}.{dbutils.widgets.get('silver_schema')}.listings_silver_merged"

try:
    history_df = spark.sql(f"DESCRIBE HISTORY {TABLE_PATH}")
    versions = history_df.count()

    print(f" [PASS] ACID Verified: Table log tracks {versions} historical versions.")

    if versions > 0:
        # 1. ADVANCED TRANSACTION SUMMARY (Extracting nested JSON metrics)
        print("\n  Deep Dive: Last 3 Transactions:")
        (history_df.selectExpr(
            "version",
            "date_format(timestamp, 'yyyy-MM-dd HH:mm:ss') as time",
            "operation",
            "userName as executed_by", # Kon chala raha hai pipeline?
            "operationMetrics.numOutputRows as rows_written", # Kitni rows insert/update hui?
            "operationParameters.mode as mode"
        )
        .orderBy(col("version").desc())
        .show(3, truncate=False))

    if versions > 1:
        # 2. TIME TRAVEL & GROWTH ANALYTICS
        print(f"  Time Travel & Data Growth Analytics:")
        current_count = spark.table(TABLE_PATH).count()
        
        # Time travelling to Version 0
        v0_count = spark.read.option("versionAsOf", 0).table(TABLE_PATH).count()
        growth = current_count - v0_count
        
        print(f"    - Version 0 (Creation) : {v0_count:,} records")
        print(f"    - Current Version ({versions-1}) : {current_count:,} records")
        print(f"    - Net Data Growth      : {growth:,} records {' ' if growth >= 0 else ' '}")

    # 3. MAINTENANCE & PERFORMANCE CHECK
    maintenance_runs = history_df.filter(col("operation").isin("OPTIMIZE", "VACUUM")).count()
    if maintenance_runs > 0:
        print(f"\n  Maintenance Status: Healthy ({maintenance_runs} OPTIMIZE/VACUUM operations found).")
    else:
        print("\n  Maintenance Alert: No OPTIMIZE or VACUUM found. Table might become slow over time!")

except Exception as e:
    print(f"  History Audit Failed: {str(e)}")

print(f"{'='*70}")